## Middleware

Agent içinde neler olup bittiğini daha sıkı bir şekilde kontrol etmenin bir yolunu sunar. Ara katman yazılımı şu durumlar için kullanışlıdır:

* Günlük kaydı (logging), analitik ve hata ayıklama (debugging) ile aracı davranışını izlemek.

* İstemleri (prompts) dönüştürmek, araç seçimini yapmak ve çıktı biçimlendirmesini düzenlemek.

* Yeniden denemeler (retries), yedek planlar (fallbacks) ve erken sonlandırma mantığı eklemek.

* Hız sınırları (rate limits), korkuluklar (guardrails) ve kişisel verilerin (PII) tespiti uygulamak.

Middleware, iki farklı uygulama, veri tabanı veya istemci-sunucu arasındaki iletişimi sağlayan bir "yazılım köprüsü"dür. Bir yapay zeka aracısı (AI Agent) bağlamında ise middleware, kullanıcının girdisi ile yapay zekanın çıktısı arasındaki denetleme istasyonu gibidir.

Neden Kullanılır? (Daha Yakından Bakış)
Middleware'i bir restoranın mutfağı (AI modeli) ile müşterisi (Kullanıcı) arasındaki garson gibi düşünebilirsin. Garson sadece yemeği taşımaz; siparişi mutfağın anlayacağı dile çevirir, yemek çıkmadan önce son bir kontrol yapar ve gerekirse eksikleri tamamlar.


## Hooks

Middleware dünyasında Hooks (Kancalar), bir işlem akışının belirli noktalarına "takılan" ve o an müdahale etmenize izin veren özel fonksiyonlardır.

Hooks Nasıl Çalışır?
Middleware yapılarında genellikle üç ana aşamada "kanca" atabilirsiniz:

* Giriş Kancası (Pre-processing / Before Hook): İstek henüz modele veya ana fonksiyona ulaşmadan devreye girer. Kullanıcı ne sordu? Bu soruda gizli bilgi var mı? Format uygun mu?

* İşlem Kancası (On-processing / During Hook): İşlem sürerken (örneğin yapay zeka cevap üretirken) araya girer. "Token" kullanımını izlemek veya cevabı canlı (stream) olarak kontrol etmek için kullanılır.

* Çıkış Kancası (Post-processing / After Hook): Model cevabı ürettikten ama kullanıcıya göstermeden hemen önce çalışır. "Cevap kurallara uygun mu? Formatı doğru mu?" gibi kontrolleri yapar.

## 1-) Summarization Middleware

Görevi özetlemektir.

LLM (Büyük Dil Modeli) uygulamasına gönderilen uzun verileri veya geçmiş konuşmaları, ana modele ulaşmadan önce "sıkıştıran" bir denetim mekanizmasıdır.

1. Nedir?
Sisteminize gelen devasa metinleri (dokümanlar, uzun sohbet geçmişi vb.) ayıklayıp, sadece en kritik bilgileri içeren kısa bir özet haline getiren ara bir kod katmanıdır. Ana modeliniz (örneğin GPT-4) soruyu cevaplamadan önce, bu middleware içeriği onun için "önceden okur ve özetler."

2. Ne İçin Kullanılır?
Context Window (Bağlam Penceresi) Yönetimi: Modellerin bir kerede okuyabileceği kelime sınırı vardır. Middleware, bu sınırı aşmamak için eski bilgileri özetleyerek yer açar.

Maliyet Tasarrufu: LLM'ler kelime (token) başına ücret alır. 10.000 kelime yerine özetlenmiş 500 kelime göndermek faturanızı ciddi oranda düşürür.

Hız (Latency): Model ne kadar az veri okursa, o kadar hızlı cevap üretir.

3. Ne Zaman Kullanılması Mantıklıdır?
Şu durumlarda bu yapıyı kurmak "şarttır":

Uzun Sohbetler (Chatbots): Kullanıcıyla 50 mesajlık bir geçmişiniz varsa, her seferinde tüm geçmişi göndermek yerine, ilk 40 mesajı özetleyip son 10 mesajı ham haliyle tutmak için.

Büyük Doküman Analizi: Binlerce sayfalık PDF'ler üzerinde soru-cevap yaparken, ilgili bölümleri bulup özetleyerek modele sunmak için.

Bütçe Kısıtı Varsa: Eğer API maliyetleri projenin sürdürülebilirliğini tehdit ediyorsa, summarization middleware en iyi "diyet" yöntemidir.

Bu yapıyı kurarken "Recursive Summarization" (özyinelemeli özetleme) dediğimiz bir teknik kullanıyoruz.

Token sınırlarına yaklaşıldığında konuşma geçmişini otomatik olarak özetleyerek; güncel mesajları olduğu gibi korur, eski bağlamı ise sıkıştırarak saklar. Özetleme şu durumlar için kullanışlıdır:

Bağlam penceresini (context window) aşan uzun soluklu konuşmalar.

Kapsamlı geçmişe sahip çok turlu diyaloglar.

Tüm konuşma bağlamının korunmasının kritik olduğu uygulamalar.

## 2-) Model Call Limit

* Model çağrılarının sayısını sınırlar.( Aşırı maliyeti önlemek için)

In [1]:
import os 
from dotenv import load_dotenv 

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq


llm = ChatGroq(model="llama-3.3-70b-versatile")

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm, 
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [6]:
config = {"configurable": {"thread_id": "test-1"}}

In [7]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='41d9bf33-24bf-4b98-9bbc-0796301dd18b'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.012066675, 'completion_tokens_details': None, 'prompt_time': 0.001970923, 'prompt_tokens_details': None, 'queue_time': 0.090455629, 'total_time': 0.014037598}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf31c-af4a-7e32-bdcf-9a62bc19071a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='41d9bf33-24bf-4b98-9bbc-0796301dd18b'), AIMess

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


from langchain_groq import ChatGroq


llm = ChatGroq(model="llama-3.3-70b-versatile")

agent = create_agent(
    model=llm, 
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm, 
            trigger=("tokens", 550),
            keep=("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}


def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  


In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


summarizer = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 600), 
    keep=("tokens", 400),   
)

agent = create_agent(
    model=llm, 
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[summarizer]
)


config = {"configurable": {"thread_id": "test-fixed"}}


initial_messages = [
    SystemMessage(content="Sen uzman bir otel asistanısın. Her zaman arama aracını kullan."),
    HumanMessage(content="Paris'te otel bul.")
]

response = agent.invoke({"messages": initial_messages}, config=config)
response

{'messages': [SystemMessage(content='Sen uzman bir otel asistanısın. Her zaman arama aracını kullan.', additional_kwargs={}, response_metadata={}, id='45b53ebe-567b-4720-9bd6-525b5c7b713f'),
  HumanMessage(content="Paris'te otel bul.", additional_kwargs={}, response_metadata={}, id='2bf86c8f-f86f-4774-9e75-ec0bfe2cd52c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2yxv8kq4r', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 245, 'total_tokens': 260, 'completion_time': 0.034953551, 'completion_tokens_details': None, 'prompt_time': 0.0228917, 'prompt_tokens_details': None, 'queue_time': 0.094081692, 'total_time': 0.057845251}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf330-517e-76f1-bdb2-

## Fraction

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  
            keep=("fraction", 0.002),    
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4


cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000 
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~57 tokens (0.0445%), 6 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='725593fb-73d8-421b-9734-3c831ab3146e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'v8fmjnrm6', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 210, 'total_tokens': 225, 'completion_time': 0.04661921, 'completion_tokens_details': None, 'prompt_time': 0.010372236, 'prompt_tokens_details': None, 'queue_time': 0.047737556, 'total_time': 0.056991446}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf336-945d-73a1-9ed6-9703197149b6-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'v8fmjnrm6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat